# Исходные карты ambientCG

Первые шесть строк train в порядке Parquet. Яркость измеряется по шкале 0–255. Дубликаты определяются по размеру и полному RGB-содержимому внутри этих шести строк.

In [1]:
from itertools import islice
from io import BytesIO
from pathlib import Path
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset, Image as DatasetImage

repo = "dream-textures/textures-color-1k"
revision = "33c49b76fcf3a18b0d521d8e760c88d49f3e47bc"
dataset = load_dataset(repo, revision=revision, split="train", streaming=True)
rows = []
seen = {}
for i, row in enumerate(islice(dataset.cast_column("image", DatasetImage(decode=False)), 6), 1):
    asset = row["image"]
    with Image.open(BytesIO(asset["bytes"])) as im:
        rgb = im.convert("RGB")
        y = np.asarray(im.convert("L"))
        digest = sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        lo, hi = np.percentile(y, [1, 99])
        rows.append({"№": i, "Файл": asset.get("path"), "Подпись": row["text"],
                     "Каналы": im.mode, "Размер": str(im.size), "Мин.": int(y.min()), "Макс.": int(y.max()),
                     "P1": float(lo), "P99": float(hi), "Дубль строки": seen.get(digest)})
        seen.setdefault(digest, i)
assert len(rows) == 6
Path("audit.json").write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame(rows).fillna("—"))

,№,Файл,Подпись,Каналы,Размер,Мин.,Макс.,P1,P99,Дубль строки
0,1,AcousticFoam001.jpg,acoustic foam dampening sound sponge,L,"(512, 512)",27,29,28.0,28.0,—
1,2,AcousticFoam002.jpg,acoustic foam dampening sound sponge,L,"(512, 512)",28,28,28.0,28.0,—
2,3,AcousticFoam003.jpg,acoustic foam dampening sound sponge,L,"(512, 512)",27,29,28.0,28.0,—
3,4,Asphalt001.jpg,asphalt clean light modern new,RGB,"(512, 512)",62,248,100.0,194.0,—
4,5,Asphalt002.jpg,asphalt rough used,RGB,"(512, 512)",16,245,60.0,196.0,—
5,6,Asphalt003.jpg,asphalt dark flat grey road street,L,"(512, 512)",50,118,83.0,100.0,—
